In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    row_number,
    sha2,
    trim,
    when,
    regexp_replace
)
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()

In [0]:
def load_curated_restructuring_profits(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Restructuring"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "profits",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

       # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "unit_code_profits",
            "businessdate",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
        spark,
        curated_zone,  # type: ignore
        "portfolio",
        portfolio_final_filters,
        portfolio_selected_fields,
        )

        # ================== it_restructures ==================

        it_restructures_extra_filters = None

        if it_restructures_extra_filters is not None:
            it_restructures_extra_filters.append(date_filter)
            it_restructures_final_filters = " AND ".join(it_restructures_extra_filters)
        else:
            it_restructures_final_filters = date_filter  # possible need change date

        it_restructures_selected_fields = [
            "RestructuringKey",
            "ReferenceDate",
            "RestructuringSystem",
            "PortfolioKey",
            "AgreementKey",
            "NewExposureKey",
            "OldExposureKey",
            "ProfitsId",
            "VatNumber",
            "CDI",
            "ProfitsId_PrimaryOwner",
            "VatNumber_PrimaryOwner",
            "CDI_PrimaryOwner",
            "InterestExposureKey",
            "InterestFreeExposureKey",
            "RestructuringAmount",
            "Writeoff_ExposureKey",
            "Writeoff_Amount",
            "SpreadPercentage",
            "RestructuringDate",
            "RestructuringProductKey",
            "Currency",
            "RestructuringTypeCode",
            "RestructuringType",
            "BusinessDate"
        ]

        it_restructures = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "profits_it_restructures",
            it_restructures_final_filters,
            it_restructures_selected_fields,
        )

        it_restructures = strip_newlines(it_restructures, ["OldExposureKey", "RestructuringKey", "Writeoff_ExposureKey"])

        # HashKeys
        result_df_profits_primary_key_fields = [
            it_restructures.RestructuringKey,
            lit("profits"),
        ]

        # ================== Selection of Final Columns ==================
            
        result_df_profits = (
                    it_restructures.join(
                        portfolio,
                        ((portfolio["unit_code_profits"] == it_restructures["PortfolioKey"] ) & (portfolio["BusinessDate"] == it_restructures["BusinessDate"])),
                        "left",
                    )
                    .withColumn(
                        "RestructuringId",
                        sha2(concat_ws("", *result_df_profits_primary_key_fields), 256),
                    )
                    .withColumn(
                        "SpreadPercentage",
                            when(
                                regexp_replace(it_restructures.SpreadPercentage, ",", ".") == "",
                                lit(None)
                            ).otherwise(
                                regexp_replace(it_restructures.SpreadPercentage, ",", ".").cast("decimal(19,2)")
                            )
                    )
                )

        result_df_profits = result_df_profits.select(
            result_df_profits.RestructuringId,
            lit("profits").alias("SourceSystem"),
            result_df_profits.unit_code_profits.alias("PortfolioKey"),
            result_df_profits.PortfolioName,
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias("IngestionTimestamp"),
            lit(None).alias("BatchId"),
            it_restructures.RestructuringKey,
            it_restructures.AgreementKey.alias("OldAgreementKey"),
            it_restructures.AgreementKey.alias("NewAgreementKey"),
            it_restructures.OldExposureKey,
            it_restructures.NewExposureKey,
            it_restructures.Writeoff_ExposureKey.alias("WriteOffExposureKey"),
            it_restructures.InterestExposureKey,
            it_restructures.InterestFreeExposureKey,
            it_restructures.ProfitsId.alias("PartyKey"),
            it_restructures.CDI,
            #result_df_profits.profitsId_PrimaryOwner.alias("PartyKey_Primary_Owner"),
            #it_restructures.CDI_PrimaryOwner.alias("CDI_PrimaryOwner"),
            it_restructures.RestructuringProductKey.cast("string").alias("RestructuringProductKey"),
            lit(None).alias("RestructuringLinkType"),
            lit(None).alias("CompletionTargetDate"),
            it_restructures.RestructuringDate.cast("date").alias("RestructuringDate"),
            it_restructures.RestructuringTypeCode.cast("string").alias("RestructuringTypeCode"),
            it_restructures.RestructuringType.cast("string").alias("RestructuringType"),
            it_restructures.RestructuringAmount.cast("decimal(19,2)").alias("RestructuringAmount"),
            it_restructures.Writeoff_Amount.cast("decimal(19,2)").alias("WriteOffAmount"),
            lit(None).alias("RestructuringDiscountAmount"),
            result_df_profits.SpreadPercentage,#all none but taking data from column
            lit(None).alias("Comment"),
            lit(None).alias("CreatedDate"),
            lit(None).alias("CreatedBy"),
            lit(None).alias("LastUpdatedDate"),
            lit(None).alias("LastUpdatedBy"),
            it_restructures.BusinessDate,
        )
        # ================== Append ==================
        result_df_profits = (
            result_df_profits.distinct()
        )  # Eliminate duplicates in SourceSystem
        result_df_profits.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")
        print(curated_table_path)
    logging.info("We did it!")

    


In [0]:
load_curated_restructuring_profits(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)